In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 32
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)
print({"embedding_dimension": model.get_sentence_embedding_dimension()})

In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)

print({
    "emb1_shape": emb1.shape,
    "emb2_shape": emb2.shape,
    "cosine_min": float(np.min(cosine_similarity)),
    "cosine_max": float(np.max(cosine_similarity)),
})

In [ ]:
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic

results_df = df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])

score_mean = float(np.mean(predicted_score_0_5))
score_std = float(np.std(predicted_score_0_5))
label_mean = float(np.mean(labels))
label_std = float(np.std(labels))
overall_mae = float(np.mean(results_df["absolute_error"]))

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "absolute_error"]].head(10))

In [ ]:
def label_bucket(x):
    if x < 2.0:
        return "low"
    elif x < 4.0:
        return "mid"
    return "high"

results_df["label_bucket"] = results_df["label"].apply(label_bucket)

bucket_summary = (
    results_df.groupby("label_bucket", sort=False)
    .agg(
        num_examples=("label", "size"),
        label_mean=("label", "mean"),
        prediction_mean=("predicted_score_0_5", "mean"),
        mae=("absolute_error", "mean"),
    )
    .reindex(["low", "mid", "high"])
    .reset_index()
)

print(bucket_summary)

In [ ]:
top_k = 5
example_columns = [
    "sentence1",
    "sentence2",
    "label",
    "predicted_score_0_5",
    "cosine_similarity",
    "absolute_error",
    "label_bucket",
]

for bucket in ["low", "mid", "high"]:
    subset = results_df[results_df["label_bucket"] == bucket]
    print(f"Top {top_k} highest-error examples for bucket={bucket}:")
    print(subset.nlargest(top_k, "absolute_error")[example_columns].reset_index(drop=True).to_string(index=False))
    print()
    print(f"Top {top_k} lowest-error examples for bucket={bucket}:")
    print(subset.nsmallest(top_k, "absolute_error")[example_columns].reset_index(drop=True).to_string(index=False))
    print("\n" + "-" * 120 + "\n")

In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"overall_mae: {overall_mae:.6f}")
print(f"predicted_score_mean: {score_mean:.6f}")
print(f"predicted_score_std: {score_std:.6f}")
print(f"label_mean: {label_mean:.6f}")
print(f"label_std: {label_std:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
print()
print("bucketed_mae_summary:")
print(bucket_summary.to_string(index=False))